[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLGlobalHealth/cntmosaic/blob/main/docs/source/tutorials/03_Tutorial_Prem_Context.ipynb)


# Estimating Context-Specific Contact Matrices with Prem's Model

Social contact patterns differ sharply by where the contact happens. Contacts at home
are dominated by partners and children; contacts at school are almost
entirely between children of the same age; contacts at work are more diverse. Many
epidemiological questions (school closures, workplace distancing, shielding the elderly) act on a single setting at a time, so modellers need contact matrices split by context.

This tutorial estimates home, school, work, and "other" contact matrices for the United
Kingdom using the `Prem2` model, `cntmosaic`'s reformulation of the approach in
[Prem, Cook & Jit (2017)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005697). The data come from the UK part of the POLYMOD survey
([Mossong et al., 2008](https://doi.org/10.1371/journal.pmed.0050074)), which is shipped with `cntmosaic`.

**Technical note.** `Prem2` is a reformulation of Prem et al.'s original estimator: it uses an intrinsic Gaussian Markov random field with a penalised-complexity prior on its precision. [Section 5](#5-how-prem2-estimates-a-contact-matrix) explains what it does.

**Workflow at a glance:**
1. Load and inspect the POLYMOD UK survey data
2. Preprocess: assign each contact to one setting and bin ages into 5-year groups
3. Understand what `Prem2` estimates
4. Fit one `Prem2` model per contact setting (home, school, work, other) with MCMC
5. Check MCMC convergence
6. Compare the four context-specific matrices

## Contents
1. [Background: setting-specific contact matrices](#1-background-setting-specific-contact-matrices)
2. [Setup](#2-setup)
3. [The POLYMOD UK data](#3-the-polymod-uk-data)
4. [Preprocessing](#4-preprocessing)
5. [How `Prem2` estimates a contact matrix](#5-how-prem2-estimates-a-contact-matrix)
6. [Fitting the four context-specific matrices](#6-fitting-the-four-context-specific-matrices)
7. [Comparing contexts](#7-comparing-contexts)
8. [Summary and next steps](#8-summary-and-next-steps)

## 1. Background: setting-specific contact matrices

A person's total contact pattern is the sum of the contacts they make in different
**settings**: at home, at school, at work, and everywhere else. Writing $m^{\text{total}}_{cd}$
for the overall contact intensity from age group $c$ to age group $d$, the decomposition introduced by
[Prem, Cook & Jit (2017)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005697)
is $m^{\text{total}}_{cd} \approx m^{\text{home}}_{cd} + m^{\text{school}}_{cd} + m^{\text{work}}_{cd} + m^{\text{other}}_{cd}$.
This is what lets transmission models represent interventions that target one setting -
closing schools zeroes out $m^{\text{school}}_{cd}$, workplace distancing scales down
$m^{\text{work}}_{cd}$, and so on.

**Setting is a property of the contact, not the respondent.** A survey respondent is not
"in" a setting - instead, each contact they report is tagged with where it took place
(the setting flags `cnt_home`, `cnt_school`, ... in POLYMOD). So setting cannot be handled as a respondent
stratification variable the way sex or region can. The standard approach, which we follow
here, is:

- take the subset of contacts that occurred in a given setting,
- fit a contact matrix to that subset,
- always divide by the **same** full set of survey respondents.

Because every setting-specific matrix uses the same respondent denominator $N_c$ (the
number of respondents in age group $c$), the four matrices are - up to the smoothing each
fit applies independently - additive, and each cell reads as "mean number of contacts of
that type, per respondent."

> **Note:** `Prem2` fits each setting independently and does **not** enforce reciprocity
> (the constraint that total contacts from group $c$ to group $d$ equal those from $d$ to
> $c$). Reciprocity adjustment needs population age counts, which we do not use in this
> tutorial; see [Summary and next steps](#8-summary-and-next-steps).

## 2. Setup

### 2.1 Running on Google Colab

If you opened this notebook in Colab, run the cell below once to install cntmosaic from GitHub. When run locally it does nothing.

In [ ]:
# @title Setup (Google Colab only)
# Installs cntmosaic from GitHub when the notebook runs in Google Colab.
# Does nothing when run locally.
import subprocess
import sys

try:
    import google.colab  # noqa: F401
except ImportError:
    pass
else:
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--quiet",
            "git+https://github.com/MLGlobalHealth/cntmosaic.git",
        ]
    )

### 2.2 Imports and JAX configuration

Two configuration lines matter for the MCMC we run later:

- `jax.config.update("jax_enable_x64", True)` enables 64-bit floating point, which the NUTS
  sampler needs for numerical stability.
- `numpyro.set_host_device_count(4)` lets the 4 MCMC chains run in parallel on CPU.

> **Note on parallel chains:** without `set_host_device_count`, JAX sees a single CPU device
> and NumPyro prints `There are not enough devices to run parallel chains: expected 4 but
> got 1` and runs the chains one after another. The line must come before the first JAX
> operation. On a GPU runtime you can leave it out.

In [ ]:
import numpy as np
import pandas as pd

import jax
from jax.random import PRNGKey

jax.config.update("jax_enable_x64", True)

import numpyro

numpyro.set_host_device_count(4)  # run the 4 MCMC chains in parallel

# cntmosaic: the data, data containers, the model, its summariser, the mosaic plot, age bins
from cntmosaic.datasets import load_polymod_uk
from cntmosaic.dataloader import ParticipantData, ContactData
from cntmosaic.models import Prem2
from cntmosaic.analysis import ModelSummariserPrem2
from cntmosaic.vis import plot_mosaic_pixilated
from cntmosaic.utils import AgeGroupSpecs

import altair as alt

## 3. The POLYMOD UK data

POLYMOD is a social contact survey conducted in eight European countries in 2005-2006
([Mossong et al., 2008](https://doi.org/10.1371/journal.pmed.0050074)). Its UK part, with
966 respondents, is shipped with `cntmosaic` and loaded with `load_polymod_uk()`, which
returns a dictionary of three tables. We use two of them:

| Table          | One row per       | Key columns                                       |
|----------------|-------------------|---------------------------------------------------|
| `participants` | survey respondent | `part_id`, `part_age`                             |
| `contacts`     | reported contact  | `part_id`, `cnt_age_exact`, and the setting flags |

Each contact links back to its respondent through `part_id`. Its setting is recorded by six
boolean flags: `cnt_home`, `cnt_school`, `cnt_work`, `cnt_transport`, `cnt_leisure` and
`cnt_otherplace`.

In [ ]:
survey_data = load_polymod_uk()
df_resp = survey_data["participants"]
df_cnt = survey_data["contacts"]

print(f"Respondents: {df_resp.shape[0]:,} rows x {df_resp.shape[1]} cols")
print(f"Contacts:    {df_cnt.shape[0]:,} rows x {df_cnt.shape[1]} cols")

Preview the respondent table - one row per person, with their age:

In [ ]:
df_resp[["part_id", "part_age"]].head()

And the contact table - one row per reported contact, with the contact's age and the
setting flags:

In [ ]:
setting_flags = ["cnt_home", "cnt_school", "cnt_work", "cnt_transport", "cnt_leisure", "cnt_otherplace"]
df_cnt[["part_id", "cnt_age_exact"] + setting_flags].head()

### Contacts by setting

The flags are not mutually exclusive: a contact can be recorded in several settings, for
example at home and at school, and a few contacts have no setting recorded at all. The
cell below counts the contacts flagged in each setting, and the number of flags per contact.

In [ ]:
print("Contacts flagged in each setting:")
print(df_cnt[setting_flags].sum().to_string())

print("\nNumber of setting flags per contact:")
print(df_cnt[setting_flags].sum(axis=1).value_counts().sort_index().to_string())

## 4. Preprocessing

`Prem2` needs two tidy tables:

- **respondents** - one row per respondent, with an age group;
- **contacts** - one row per contact, with an age group and (for us) the setting.

### 4.1 Assign each contact to one setting

Each setting-specific matrix should count every contact exactly once, so we assign each
contact to a single setting by the hierarchy **Home > School > Work > Other**: a contact
goes to the first of its flags in this order. Contacts at transport, leisure or other places,
and contacts with no setting recorded, are assigned to **Other**. The four settings then
partition the contacts, and the four matrices add up to the matrix of all contacts.

We then keep the identifier, age and setting columns, and cast ages to a nullable integer
type. Following cntmosaic's naming convention, respondent columns are prefixed `part_` and
contact columns `cnt_`.

In [ ]:
# One setting per contact: Home > School > Work > Other
df_cnt = df_cnt.assign(
    cnt_setting=np.select(
        [df_cnt["cnt_home"], df_cnt["cnt_school"], df_cnt["cnt_work"]],
        ["Home", "School", "Work"],
        default="Other",
    )
)
print(df_cnt["cnt_setting"].value_counts().to_string())

# Respondents: keep id + age; contacts: keep id + age + setting
df_resp_slim = df_resp[["part_id", "part_age"]].copy()
df_cnt_slim = df_cnt[["part_id", "cnt_age_exact", "cnt_setting"]].copy()

# Cast ages to a nullable integer type
df_resp_slim["part_age"] = df_resp_slim["part_age"].astype("Int64")
df_cnt_slim["cnt_age"] = df_cnt_slim["cnt_age_exact"].astype("Int64")

### 4.2 Bin ages into five-year groups

`AgeGroupSpecs(min=0, max=79, step=5)` defines sixteen 5-year bins - `[0, 5)`, `[5, 10)`,
..., `[75, 80)` - where `min` and `max` are *inclusive* age bounds. Its `.cut()` method
turns a column of integer ages into the matching interval labels; both the model and
`plot_mosaic_pixilated` read these bins.

The oldest POLYMOD UK respondent is 79, so we stop at 79: an `[80, 85)` group would have no
respondents, and `Prem2` raises an error if any respondent age group is empty. We then
restrict both tables to ages 0-79, which drops the contacts aged 80 or over and those
without an exact age.

In [ ]:
# Five-year age groups: [0, 5), [5, 10), ..., [75, 80)
age_grp_specs = AgeGroupSpecs(min=0, max=79, step=5)
df_resp_slim["part_age_grp"] = age_grp_specs.cut(df_resp_slim["part_age"])
df_cnt_slim["cnt_age_grp"] = age_grp_specs.cut(df_cnt_slim["cnt_age"])

# Keep ages 0-79 only (drops out-of-range / missing ages so no age bin is empty)
df_resp_slim = df_resp_slim[df_resp_slim["part_age"].between(0, 79)].copy()
df_cnt_slim = df_cnt_slim[df_cnt_slim["cnt_age"].between(0, 79)].copy()

print(f"{len(df_resp_slim)} respondents, {len(df_cnt_slim)} contacts")

## 5. How `Prem2` estimates a contact matrix

For a single setting, the data reduce to two arrays over the 16 age groups:

- $Y_{cd}$ - the number of contacts that respondents in age group $c$ reported with people
  in age group $d$;
- $N_c$ - the number of respondents in age group $c$.

The quantity we want is the **contact intensity** $m_{cd}$: the mean number of contacts (in
this setting) that a respondent aged $c$ has with people aged $d$. The obvious estimate
$Y_{cd}/N_c$ is very noisy wherever a cell has few respondents or few contacts - exactly
where survey data are thinnest (young-old cells, the oldest ages).

`Prem2` fits a smooth surface through the log-intensity instead. With the default
negative-binomial likelihood the model is $Y_{cd} \sim \text{NegBin}(\mu_{cd},\,\phi)$
with $\mu_{cd} = N_c \exp(\beta_0 + f_{cd})$, so the contact intensity is
$m_{cd} = \exp(\beta_0 + f_{cd})$. Here $\beta_0$ is a single baseline level and $f$ is a
16x16 surface of age-by-age deviations from it (only their sum $\beta_0 + f_{cd}$ is
identified by the data - [Section 6.5](#65-checking-convergence) returns to this). The $N_c$
term enters as a fixed offset, so $Y$ is modelled as counts while $m_{cd}$ is a per-respondent
rate.

**The smoothing prior.** $f$ is given a first-order 2-D *intrinsic Gaussian Markov random
field* (IGMRF) prior. Informally, this penalises the squared difference between neighbouring
cells of $f$ in both the respondent-age and contact-age directions - a 2-D random walk - so
each cell borrows strength from its neighbours and the estimated surface is smooth. A single
precision parameter $\tau$ controls how smooth: `Prem2` places a *penalised-complexity*
prior on $\tau$ (equivalent to $\tau^{-1/2} \sim \text{Exponential}(2.86)$ on the marginal
standard deviation), which shrinks towards the smooth solution unless the data clearly
support more structure.

|             | Prem et al. (2017)             | `Prem2` (this notebook)                |
|-------------|--------------------------------|----------------------------------------|
| Inference   | plug-in estimate + bootstrap   | full Bayesian (NUTS or SVI)            |
| Smoothing   | applied post hoc               | IGMRF prior inside the model           |
| Likelihood  | -                              | negative binomial (default) or Poisson |
| Uncertainty | bootstrap resampling           | posterior credible intervals           |
| Reciprocity | enforced using population data | not enforced (no population data used) |

> **Note:** the original `Prem` model in cntmosaic fits the same idea to individual contact
> records and can add respondent-level random effects. `Prem2` works with aggregated count
> cells, is faster, and is the recommended starting point.

## 6. Fitting the four context-specific matrices

We now fit one `Prem2` model per setting. The pattern is the same each time:

1. wrap the respondents in a `ParticipantData` container (built **once**, reused for every
   setting) and the setting's contacts in a `ContactData` container;
2. construct `Prem2` and run MCMC;
3. wrap the result in `ModelSummariserPrem2` to get posterior summaries.

`ParticipantData` and `ContactData` validate their inputs (checking column types, dropping
rows with missing ages) and standardise column names for the model.

### 6.1 Home

`cnt_hm_data` is just the `cnt_setting == "Home"` slice of the contacts; `part_data`
stays the full respondent set.

In [ ]:
part_data = ParticipantData(
    data=df_resp_slim, id_col="part_id", age_grp_col="part_age_grp"
)

# Home contacts
cnt_hm_data = ContactData(
    data=df_cnt_slim[df_cnt_slim["cnt_setting"] == "Home"],
    id_col="part_id",
    age_grp_col="cnt_age_grp",
)

`run_inference_mcmc` runs the No-U-Turn Sampler (NUTS). The main arguments:

| Argument             | Meaning                           | Value here    |
|----------------------|-----------------------------------|---------------|
| `prng_key`           | JAX random seed (reproducibility) | `PRNGKey(1)`  |
| `num_warmup`         | adaptation steps, discarded       | 500           |
| `num_samples`        | posterior draws kept per chain    | 1000          |
| `num_chains`         | independent chains                | 4             |
| `target_accept_prob` | NUTS target acceptance rate       | 0.8 (default) |
| `max_tree_depth`     | maximum NUTS trajectory length    | 10 (default)  |

After sampling, the model prints `Number of divergences: 0` (and again as
`Number of divergent transitions: 0`). Zero is what we want; a non-zero count means the
sampler struggled with the posterior geometry and the estimates may be biased - we hit
exactly this for the school model in
[Section 6.2](#62-school-responding-to-divergences).

Constructing `ModelSummariserPrem2(prem_hm)` prints a warning:
`PopulationData not provided. Reciprocity adjustment and depixilation will not be possible.`
**This is expected.** We fit contact intensities only and do not pass population data, so
contact *rates*, reciprocity adjustment, and 1-year-resolution matrices are unavailable. The
contact-*intensity* matrix we plot here needs none of them.

> **Note:** MCMC gives exact posterior samples but is the slow option. For quick
> exploration, `prem_hm.run_inference_svi(PRNGKey(0), num_steps=20_000)` runs variational
> inference instead; everything downstream (`ModelSummariserPrem2`, the plots) is
> identical.

In [ ]:
# Initialize the Prem2 model
prem_hm = Prem2(
    part_data=part_data, cnt_data=cnt_hm_data, age_group_specs=age_grp_specs
)

# Run MCMC inference
prng_key = PRNGKey(1)
prem_hm.run_inference_mcmc(prng_key, num_warmup=500, num_samples=1000, num_chains=4)

# Summarize the results
summariser_hm = ModelSummariserPrem2(prem_hm)

`summarise_cint()` returns a dictionary of posterior summaries keyed by `"source->target"`
stratum. This model is unstratified, so there is a single key, `"All->All"`. Each value is a
`ContactSummary` with `.central` (the posterior median matrix), `.lower`, and `.upper` (the
95% credible interval), and it can be passed straight to `plot_mosaic_pixilated`.

`plot_mosaic_pixilated` draws the matrix at age-group ("pixel") resolution: each block is
one age-group pair, block size shows the bin width, colour shows the intensity, and the
y-axis runs bottom-to-top from youngest to oldest.

In [ ]:
summ_cint_hm = summariser_hm.summarise_cint()
plot_mosaic_pixilated(
    summ_cint_hm["All->All"],
    age_grp_specs,
    xlabel="Age of respondent",
    ylabel="Age of contact",
    zlabel="Intensity",
    title="Home Contacts",
    color_min=0.0,
    color_max=1.0,
)

**Interpreting the results:**
- **Strong leading diagonal**: assortative mixing: partners, siblings and similarly-aged
  household members.
- **Two off-diagonal bands ~30 years off the diagonal**: the parent-child relationship,
  seen from both sides (a 35-year-old with a 5-year-old, and vice versa).

### 6.2 School: responding to divergences

School contacts are concentrated in a handful of child age cells, which gives the posterior
a sharp, awkward geometry. With the default settings, NUTS reports **divergent transitions**
for this model (32 in our run) and warns
`Found N divergent transitions. Consider increasing target_accept_prob or max_tree_depth.`

The fix, already applied in the cell below, is to make the sampler take smaller, more
careful steps: `target_accept_prob=0.98` (from 0.8) and `max_tree_depth=13` (from 10). This
removes the divergences at the cost of a slower run. Raising these two settings is the
standard first response to divergences.

In [ ]:
cnt_sc_data = ContactData(
    data=df_cnt_slim[df_cnt_slim["cnt_setting"] == "School"],
    id_col="part_id",
    age_grp_col="cnt_age_grp",
)

prem_sc = Prem2(
    part_data=part_data, cnt_data=cnt_sc_data, age_group_specs=age_grp_specs
)

# Tighter NUTS settings than the default to remove divergent transitions:
# smaller steps (target_accept_prob) and longer trajectories (max_tree_depth).
prng_key = PRNGKey(1)
prem_sc.run_inference_mcmc(
    prng_key,
    num_warmup=500,
    num_samples=1000,
    num_chains=4,
    target_accept_prob=0.98,
    max_tree_depth=13,
)

summariser_sc = ModelSummariserPrem2(prem_sc)

In [ ]:
summ_cint_sc = summariser_sc.summarise_cint()
plot_mosaic_pixilated(
    summ_cint_sc["All->All"],
    age_grp_specs,
    xlabel="Age of respondent",
    ylabel="Age of contact",
    zlabel="Intensity",
    title="School Contacts",
    color_min=0.0,
    color_max=4.0,
)

**Interpreting the results:**
- **Dominant diagonal at ages 5-19**: classmates of the same age.
- **Faint band of contacts aged ~25-50 in the children's rows**: teachers and school staff.
- **Near-zero everywhere else**: few school contacts outside education ages.

### 6.3 Work: when there are no divergences but mixing is poor

Work contacts fill in the adult ages. With the default settings this model reports no
divergences, but the chains mix poorly: the effective sample sizes of the smoothing
precision `tau` are low (a tail ESS of about 100 from 4,000 draws) and its `r_hat` is 1.02.
The absence of divergences alone does not show that the sampler has explored the posterior
well, so the convergence checks of [Section 6.5](#65-checking-convergence) are still needed.

Raising `target_accept_prob` to 0.95 fixes this: the effective sample sizes of `tau` rise
above 800, and `r_hat` falls to 1.01.

In [ ]:
cnt_wk_data = ContactData(
    data=df_cnt_slim[df_cnt_slim["cnt_setting"] == "Work"],
    id_col="part_id",
    age_grp_col="cnt_age_grp",
)

prem_wk = Prem2(
    part_data=part_data, cnt_data=cnt_wk_data, age_group_specs=age_grp_specs
)

prng_key = PRNGKey(1)
# A higher target acceptance rate (smaller steps) improves mixing for this model.
prem_wk.run_inference_mcmc(
    prng_key,
    num_warmup=500,
    num_samples=1000,
    num_chains=4,
    target_accept_prob=0.95,
)

summariser_wk = ModelSummariserPrem2(prem_wk)

In [ ]:
summ_cint_wk = summariser_wk.summarise_cint()
plot_mosaic_pixilated(
    summ_cint_wk["All->All"],
    age_grp_specs,
    xlabel="Age of respondent",
    ylabel="Age of contact",
    zlabel="Intensity",
    title="Work Contacts",
    color_min=0.0,
    color_max=0.8,
)

**Interpreting the results:**
- **Broad working-age block (~20-60)** - colleagues span a wide age range, so the pattern is
  much less diagonal than school.
- **Empty rows and columns for children and the retired** - as expected.

### 6.4 Other

"Other" collects every contact that was not assigned to home, school, or work: contacts
at transport, leisure and other places, and contacts with no setting recorded.

In [ ]:
cnt_ot_data = ContactData(
    data=df_cnt_slim[df_cnt_slim["cnt_setting"] == "Other"],
    id_col="part_id",
    age_grp_col="cnt_age_grp",
)

prem_ot = Prem2(
    part_data=part_data, cnt_data=cnt_ot_data, age_group_specs=age_grp_specs
)

prng_key = PRNGKey(1)
prem_ot.run_inference_mcmc(prng_key, num_warmup=500, num_samples=1000, num_chains=4)

summariser_ot = ModelSummariserPrem2(prem_ot)

In [ ]:
summ_cint_ot = summariser_ot.summarise_cint()
plot_mosaic_pixilated(
    summ_cint_ot["All->All"],
    age_grp_specs,
    xlabel="Age of respondent",
    ylabel="Age of contact",
    zlabel="Intensity",
    title="Other Contacts",
    color_min=0.0,
    color_max=1.6,
)

**Interpreting the results:**
- **Strongest diagonal among teenagers and young adults (~10-24)** - friends and leisure
  contacts of the same age.
- **Broad off-diagonal support** - casual contacts in public spaces are less age-assortative
  than home or school contacts.

### 6.5 Checking convergence

Before trusting any of these matrices, check that the four chains actually converged.
`prem_hm._mcmc_result` is the underlying NumPyro `MCMC` object, which
[ArviZ](https://python.arviz.org) can read directly (the feature-selection tutorial,
`04_Tutorial_Feature_Selection.ipynb`, surfaces the same object as `result.idatas`).

We monitor `log_cint` - the log contact-intensity surface $\beta_0 + f$ that the matrix is
built from - together with the scalars `tau` (smoothing precision) and `inv_disp`
(dispersion). We deliberately do **not** check `baseline` and `f` separately: the IGMRF
prior does not pin the overall level of `f`, and the likelihood only ever sees the sum
$\beta_0 + f_{cd}$, so those two are not individually identified. Only `log_cint` is, and
that is all the matrix depends on. `az.summary(idata_hm, var_names=["baseline"])` on its own
will look unconverged; that is expected and harmless here.

In [ ]:
import arviz as az

idata_hm = az.from_numpyro(prem_hm._mcmc_result)

# tau is the smoothing precision; inv_disp is the negative-binomial dispersion.
# Want r_hat close to 1.00 and ess in the hundreds or more.
az.summary(idata_hm, var_names=["tau", "inv_disp"])

Both should have `r_hat` close to 1.00 (below 1.01) and `ess_bulk` / `ess_tail` (effective
sample sizes) of at least a few hundred - enough for stable medians and 95% intervals.

`log_cint` has 256 cells, so summarise its diagnostics rather than printing every row:

In [ ]:
# log_cint = baseline + f is the log contact-intensity surface (256 cells) - the
# quantity the matrix is actually built from. Summarise its diagnostics rather
# than printing every row.
az.summary(idata_hm, var_names=["log_cint"])[["r_hat", "ess_bulk"]].describe()

> **Note:** run the same check on `prem_sc`, `prem_wk`, and `prem_ot`. The school and work
> models are the ones to watch, since they needed tighter sampler settings.

## 7. Comparing contexts

Finally, place the four posterior-median matrices side by side.

**A note on colour scales.** The cell below gives each panel its **own** colour scale
(`resolve_scale(color="independent")`). This makes the structure within each setting easy to
see, but it hides the large differences in magnitude between settings - the school maximum
is several times the others. To compare magnitudes directly, set a single shared
`color_min` / `color_max` across all four panels and drop `resolve_scale`.

In [ ]:
charts = {}
for i, (title, summ, vmax) in enumerate(
    [
        ("Home", summ_cint_hm, 1.0),
        ("School", summ_cint_sc, 4.0),
        ("Work", summ_cint_wk, 0.8),
        ("Other", summ_cint_ot, 1.6),
    ]
):
    charts[title] = plot_mosaic_pixilated(
        summ["All->All"],
        age_grp_specs,
        xlabel="Age of respondent",
        ylabel="Age of contact",
        zlabel="Intensity",
        title=title,
        show_y_label=(i % 2 == 0),  # left column
        show_y_ticks=(i % 2 == 0),
        show_x_label=(i >= 2),  # bottom row
        show_x_ticks=(i >= 2),
        width=200,
        height=200,
        color_min=0.0,
        color_max=vmax,
    )

top_row = alt.hconcat(charts["Home"], charts["School"])
bottom_row = alt.hconcat(charts["Work"], charts["Other"])
final_chart = alt.vconcat(top_row, bottom_row).resolve_scale(color="independent")

display(final_chart)

**Interpreting the results:**
- **Home** - a diagonal plus parent-child bands.
- **School** - a spike on the child diagonal, near-zero elsewhere, and by far the highest
  intensities.
- **Work** - a broad adult block, weakly diagonal.
- **Other** - strongest among teenagers and young adults, otherwise diffuse.

Each cell is a mean number of contacts per respondent in that setting, so - up to the
smoothing each model applied independently - the four matrices sum to the all-contexts
contact matrix.

## 8. Summary and next steps

In this tutorial you:
- loaded and inspected the POLYMOD UK contact survey;
- assigned each contact to one setting, binned ages into 5-year groups, and split contacts by setting;
- fitted one `Prem2` model per setting (home, school, work, other) with MCMC,
  tightening the sampler for the school model to remove divergences, and for the work
  model to improve mixing;
- checked convergence with ArviZ (`r_hat`, effective sample size);
- compared the four context-specific contact matrices.

**Where to go next:**
- **Add population data** - pass a `PopulationData` to `ModelSummariserPrem2` (for the UK,
  the `population` table returned by `load_polymod_uk()`, summed over gender) to unlock reciprocity
  adjustment (`apply_reciprocity=True`), contact **rates** (`summarise_rate`),
  and 1-year-resolution matrices (`summarise_cint(return_depixilated=True)`).
- **Stratify further** - add respondent sex via `strat_var_cols` on
  `ParticipantData`; see `01_Tutorial_GenMix.ipynb`.
- **Faster inference** - swap `run_inference_mcmc` for `run_inference_svi`.
- **Other smoothing priors** - `PSpline2D`, `HSGP2D`, `vdKassteele2D` in place of the IGMRF.
- **API reference** - `Prem2`, `ModelSummariserPrem2`, `plot_mosaic_pixilated`.